In [ ]:
# da usare se le librerie usate non sono installate
!pip install requests
!pip install bs4
!pip install selenium
!pip install chromedriver-autoinstaller

In [ ]:
from selenium import webdriver
import chromedriver_autoinstaller
import requests
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import pandas as pd

In [ ]:
# setup chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless') # ensure GUI is off
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# set path to chromedriver as per your configuration
chromedriver_autoinstaller.install()

# set up the webdriver
driver = webdriver.Chrome(options=chrome_options)

In [ ]:
titles = []
reviews = []
prices = []
availabilities = []
reviews_values = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

In [ ]:
# itero su tutte le pagine del catalogo
for i in range(1,51):
  url = f'https://books.toscrape.com/catalogue/page-{i}.html'
  print(f"Scansiono la pagina num. {i}")
  driver.get(url)
  response = requests.get(url)
  try:
    response.raise_for_status()
    # aspetto che si veda il body
    WebDriverWait(driver,10).until(EC.visibility_of_element_located((By.ID, 'default')))
    products_dom = driver.find_elements(By.CLASS_NAME, 'product_pod')

    for product in products_dom:
      try:
        title = product.find_element(By.TAG_NAME, 'a').accessible_name
        # estraggo il numero di stelle, dal valore dell'attirbuto class del tag p
        reviews_value = product.find_element(By.TAG_NAME, 'p').get_attribute('class').split()[-1]
        price = product.find_element(By.CLASS_NAME, 'price_color').text
        availability = product.find_element(By.XPATH, "//div[@class='product_price']/p[@class='instock availability']").text
        # se non ho riscontrato errori, posso salvare il tutto
        titles.append(title)
        reviews.append(reviews_values[reviews_value])
        prices.append(price)
        availabilities.append(availability)
      except:
        print("Errore nel recupero dei dati del libro, passo al libro successivo")
        continue
  except HTTPError:
    print("Errore nell'elaborazione della richiesta, riprova più tardi")

  # inserisco una piccola pausa per evitare di fare troppe richieste in breve tempo
  sleep(3)


In [ ]:
# ci sono 20 libri in ogni pagina, per 50 pagine. Verifico che gli array siano della lunghezza giusta
print(len(titles), len(reviews), len(prices), len(availabilities))

1000 1000 1000 1000


In [ ]:
df = pd.DataFrame({'Titolo': titles, 'Valutazione in stelle' : reviews, 'Prezzo': prices, 'Disponibilità' : availabilities})
df

,Titolo,Valutazione in stelle,Prezzo,Disponibilità
0,A Light in the Attic,3,£51.77,In stock
1,Tipping the Velvet,1,£53.74,In stock
2,Soumission,1,£50.10,In stock
3,Sharp Objects,4,£47.82,In stock
4,Sapiens: A Brief History of Humankind,5,£54.23,In stock
...,...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,1,£55.53,In stock
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",4,£57.06,In stock
997,A Spy's Devotion (The Regency Spies of London #1),5,£16.97,In stock
998,1st to Die (Women's Murder Club #1),1,£53.98,In stock


In [ ]:
# esportazione in csv
df.to_csv('books.csv', index=False)